# CSA0812 – Assignment 5
## B2 – Engineer Adversarial Data

**Student:** Darshan A  
**Register:** 1973260011  
**Marks:** 5 | **K4 – Analyse**

## B2 – Engineer Adversarial Data (5 Marks)

This task deliberately tests inputs that can break a normal CSV workflow:
- A file with a missing column.
- A file using a semicolon instead of a comma.
- A Latin-1 encoded file read as UTF-8.
- An empty file.
- A station name containing an apostrophe.
- Sentences with sarcasm, double negatives and mixed opinions.

### Adversarial file results and what they reveal

| Input | Expected / observed response | Design lesson |
|---|---|---|
| Missing `pm25` column | Report missing required column | Validate the header before processing rows |
| Wrong delimiter | Header does not contain expected fields | Detect delimiter/schema mismatch |
| Latin-1 text read as UTF-8 | Catch `UnicodeDecodeError` | Encoding must be known or handled explicitly |
| Empty file | Report empty input rather than crashing | Check file size/header/rows |
| Quoted station `O'Harbor` | String concatenation breaks SQL syntax; parameter placeholder works | Use parameterised SQL |
| Sarcasm | May be incorrectly labelled | Lexicon rules do not understand context |
| Double negative | Simple previous-word negation may flip only one word and misread meaning | Need contextual language processing |
| Mixed opinions | Positive and negative scores can cancel | A single score may hide two separate aspects |

### Complete the Code – TODO 1 and TODO 2

TODO 1 inserts every reading using one `executemany()` call and `?` placeholders, then commits. TODO 2 returns a station average using a parameterised `SELECT ... WHERE station = ?`.

The expected output is:

```text
54.0 108.0
None
```

The attempted injection is treated as a literal station name. Since no station has that exact name, the query returns `None` rather than all rows.

### B2 Complete the Code – Executed

In [3]:
"""B2 Complete the Code – TODO 1 and TODO 2 completed."""
import sqlite3

readings = [
    ("Adyar", "2026-10-01", 48.0),
    ("Guindy", "2026-10-01", 95.0),
    ("Guindy", "2026-10-03", 121.0),
    ("Adyar", "2026-10-04", 60.0),
]

con = sqlite3.connect(":memory:")
con.execute("CREATE TABLE aq (station TEXT, day TEXT, pm25 REAL)")

# TODO 1 completed: one executemany call with ? placeholders, then commit.
con.executemany(
    "INSERT INTO aq(station, day, pm25) VALUES (?, ?, ?)",
    readings
)
con.commit()


def station_average(name):
    # TODO 2 completed: parameterised SELECT using WHERE station = ?.
    row = con.execute(
        "SELECT AVG(pm25) FROM aq WHERE station = ?",
        (name,)
    ).fetchone()
    return row[0]


print(station_average("Adyar"), station_average("Guindy"))
print(station_average("Adyar' OR '1'='1"))
con.close()


54.0 108.0
None


### Alternative Concept and Complexity Analysis

**String concatenation instead of a placeholder:** With `Adyar' OR '1'='1`, concatenation inserts the supplied text into SQL syntax. The `OR` expression can make the WHERE condition true for every row. An apostrophe in a legitimate station name such as `O'Harbor` can also break the SQL statement. A `?` placeholder treats the supplied text as a value, preventing it from changing the query structure.

**`executemany()` versus `execute()` in a loop:** Both insert n rows and have O(n) overall insertion work. `executemany()` reduces repeated Python-to-SQLite call overhead and is normally faster for batch inserts. Space is O(n) if the entire batch is built in memory; batching smaller chunks limits the working batch size.

### Executable adversarial test suite

The code below creates the four malformed files, inspects them, demonstrates unsafe versus parameterised SQL, compares `executemany()` with a loop, and records the lexicon scorer's failures.

In [4]:
"""
Assignment 5 – B2: Engineer Adversarial Data
Darshan A | 1973260011 | CSA0812

Run:
    python B2_adversarial_data.py

Creates malformed CSV test files, compares unsafe and parameterised SQL,
completes the supplied executemany/parameterised SELECT TODOs, measures
executemany versus execute-in-a-loop, and tests a simple lexicon scorer
against sarcasm, double negatives and mixed opinions.
"""

from pathlib import Path
import csv
import sqlite3
import time
import statistics

OUT = Path("assignment5_b2_output")
OUT.mkdir(exist_ok=True)

# ---------------------------------------------------------------------
# COMPLETE THE CODE – TODO 1 and TODO 2 from the assignment
# ---------------------------------------------------------------------

readings = [
    ("Adyar", "2026-10-01", 48.0),
    ("Guindy", "2026-10-01", 95.0),
    ("Guindy", "2026-10-03", 121.0),
    ("Adyar", "2026-10-04", 60.0),
]


def complete_code_demo():
    con = sqlite3.connect(":memory:")
    con.execute("CREATE TABLE aq (station TEXT, day TEXT, pm25 REAL)")

    # TODO 1 completed: one executemany call with ? placeholders, then commit.
    con.executemany(
        "INSERT INTO aq(station, day, pm25) VALUES (?, ?, ?)",
        readings
    )
    con.commit()

    # TODO 2 completed: parameterised SELECT with WHERE station = ?.
    def station_average(name):
        row = con.execute(
            "SELECT AVG(pm25) FROM aq WHERE station = ?",
            (name,)
        ).fetchone()
        return row[0]

    print("B2 COMPLETE THE CODE")
    print(station_average("Adyar"), station_average("Guindy"))
    print(station_average("Adyar' OR '1'='1"))
    assert station_average("Adyar") == 54.0
    assert station_average("Guindy") == 108.0
    assert station_average("Adyar' OR '1'='1") is None
    con.close()


# ---------------------------------------------------------------------
# Adversarial files: missing column, wrong delimiter, other encoding, empty
# ---------------------------------------------------------------------

EXPECTED_COLUMNS = {"station", "date", "pm25"}


def create_adversarial_files():
    missing = OUT / "missing_column.csv"
    missing.write_text(
        "station,date\nAdyar,2026-10-01\n", encoding="utf-8"
    )

    wrong_delimiter = OUT / "wrong_delimiter.csv"
    wrong_delimiter.write_text(
        "station;date;pm25\nAdyar;2026-10-01;48\n", encoding="utf-8"
    )

    other_encoding = OUT / "latin1_encoding.csv"
    other_encoding.write_bytes(
        "station,date,pm25\nCafé,2026-10-01,48\n".encode("latin-1")
    )

    empty = OUT / "empty.csv"
    empty.write_text("", encoding="utf-8")
    return [missing, wrong_delimiter, other_encoding, empty]


def inspect_adversarial_file(path):
    if path.stat().st_size == 0:
        return "EMPTY FILE: no header or records"

    try:
        with open(path, "r", newline="", encoding="utf-8") as file:
            reader = csv.DictReader(file)
            columns = set(reader.fieldnames or [])
            if not EXPECTED_COLUMNS.issubset(columns):
                return f"MISSING COLUMN(S): {sorted(EXPECTED_COLUMNS - columns)}"
            first_row = next(reader, None)
            if first_row is None:
                return "HEADER FOUND, BUT NO DATA ROWS"
            return f"OK: columns found; first row={first_row}"
    except UnicodeDecodeError as error:
        return f"ENCODING ERROR: {error.__class__.__name__}"
    except csv.Error as error:
        return f"CSV ERROR: {error}"


# ---------------------------------------------------------------------
# Unsafe SQL concatenation vs safe parameter placeholders
# ---------------------------------------------------------------------

def demonstrate_sql_injection():
    con = sqlite3.connect(":memory:")
    con.execute("CREATE TABLE aq (station TEXT, day TEXT, pm25 REAL)")
    con.executemany(
        "INSERT INTO aq VALUES (?, ?, ?)",
        [
            ("Adyar", "2026-10-01", 48.0),
            ("Guindy", "2026-10-01", 95.0),
            ("O'Harbor", "2026-10-01", 77.0),
        ]
    )
    con.commit()

    malicious = "Adyar' OR '1'='1"
    unsafe_query = (
        "SELECT station, pm25 FROM aq WHERE station = '" + malicious + "'"
    )
    print("\nUNSAFE SQL STRING CONCATENATION")
    print(unsafe_query)
    try:
        unsafe_rows = con.execute(unsafe_query).fetchall()
        print("Rows returned by unsafe query:", unsafe_rows)
        print("The injected OR condition can make the WHERE condition true for all rows.")
    except sqlite3.Error as error:
        print("Unsafe SQL error:", error)

    safe_rows = con.execute(
        "SELECT station, pm25 FROM aq WHERE station = ?",
        (malicious,)
    ).fetchall()
    print("Rows returned by parameterised query:", safe_rows)
    assert safe_rows == []

    # A legitimate station name containing an apostrophe also works with ?.
    quote_name = "O'Harbor"
    safe_quote = con.execute(
        "SELECT station, pm25 FROM aq WHERE station = ?",
        (quote_name,)
    ).fetchall()
    print("Quoted station searched safely:", safe_quote)

    try:
        broken = con.execute(
            "SELECT station, pm25 FROM aq WHERE station = '" + quote_name + "'"
        ).fetchall()
        print("Concatenated query for O'Harbor:", broken)
    except sqlite3.Error as error:
        print("Concatenated query for O'Harbor fails:", type(error).__name__)

    con.close()


# ---------------------------------------------------------------------
# Compare executemany with execute in a loop
# ---------------------------------------------------------------------

def benchmark_insert_methods(n=5000):
    data = [
        (f"Station{i % 12}", f"2026-10-{i % 28 + 1:02d}", float(i % 200))
        for i in range(n)
    ]

    def with_executemany():
        con = sqlite3.connect(":memory:")
        con.execute("CREATE TABLE aq (station TEXT, day TEXT, pm25 REAL)")
        con.executemany("INSERT INTO aq VALUES (?, ?, ?)", data)
        con.commit()
        count = con.execute("SELECT COUNT(*) FROM aq").fetchone()[0]
        con.close()
        return count

    def with_execute_loop():
        con = sqlite3.connect(":memory:")
        con.execute("CREATE TABLE aq (station TEXT, day TEXT, pm25 REAL)")
        for row in data:
            con.execute("INSERT INTO aq VALUES (?, ?, ?)", row)
        con.commit()
        count = con.execute("SELECT COUNT(*) FROM aq").fetchone()[0]
        con.close()
        return count

    # Repeat each small benchmark three times and report medians.
    many_times = []
    loop_times = []
    for _ in range(3):
        start = time.perf_counter()
        assert with_executemany() == n
        many_times.append(time.perf_counter() - start)

        start = time.perf_counter()
        assert with_execute_loop() == n
        loop_times.append(time.perf_counter() - start)

    print("\nINSERT PERFORMANCE")
    print(f"Rows inserted: {n}")
    print(f"executemany median: {statistics.median(many_times):.4f} s")
    print(f"execute loop median: {statistics.median(loop_times):.4f} s")
    print("Both perform O(n) insert operations; executemany reduces Python-to-SQLite call overhead.")


# ---------------------------------------------------------------------
# Lexicon sentiment tests: sarcasm, double negation and mixed opinions
# ---------------------------------------------------------------------

POSITIVE = {"good", "clean", "fast", "helpful", "improved", "excellent", "safe"}
NEGATIVE = {"bad", "dirty", "slow", "rude", "smoke", "broken", "poor"}
NEGATORS = {"not", "never", "no"}


def score(text):
    words = (
        text.lower().replace(".", " ").replace(",", " ")
        .replace("!", " ").replace("?", " ").split()
    )
    total = 0
    for index, word in enumerate(words):
        value = 1 if word in POSITIVE else -1 if word in NEGATIVE else 0
        if index > 0 and words[index - 1] in NEGATORS:
            value = -value
        total += value
    return total


def sentiment_label(text):
    value = score(text)
    return "positive" if value > 0 else "negative" if value < 0 else "neutral"


def test_sentiment_adversarial_cases():
    examples = [
        "The air is not good today.",
        "The service was not not good.",
        "Wonderful! Another day of dirty air and smoke.",
        "The road is clean but the bus is slow.",
        "I never said the air was bad.",
        "The air is good, but the smell is dirty.",
    ]
    print("\nADVERSARIAL SENTIMENT TESTS")
    for text in examples:
        print(f"{sentiment_label(text):8} | {text}")
    print(
        "Limitations: sarcasm needs context; double negation is not handled reliably; "
        "mixed positive and negative opinions may cancel to neutral."
    )


def main():
    complete_code_demo()

    print("\nADVERSARIAL FILE TESTS")
    for path in create_adversarial_files():
        print(path.name, "->", inspect_adversarial_file(path))

    demonstrate_sql_injection()
    benchmark_insert_methods()
    test_sentiment_adversarial_cases()

    print("\nB2 completed: malformed files, SQL safety, insert methods and sentiment limitations tested.")


if __name__ == "__main__":
    main()


B2 COMPLETE THE CODE
54.0 108.0
None

ADVERSARIAL FILE TESTS
missing_column.csv -> MISSING COLUMN(S): ['pm25']
wrong_delimiter.csv -> MISSING COLUMN(S): ['date', 'pm25', 'station']
latin1_encoding.csv -> ENCODING ERROR: UnicodeDecodeError
empty.csv -> EMPTY FILE: no header or records

UNSAFE SQL STRING CONCATENATION
SELECT station, pm25 FROM aq WHERE station = 'Adyar' OR '1'='1'
Rows returned by unsafe query: [('Adyar', 48.0), ('Guindy', 95.0), ("O'Harbor", 77.0)]
The injected OR condition can make the WHERE condition true for all rows.
Rows returned by parameterised query: []
Quoted station searched safely: [("O'Harbor", 77.0)]
Concatenated query for O'Harbor fails: OperationalError

INSERT PERFORMANCE
Rows inserted: 5000
executemany median: 0.0063 s
execute loop median: 0.0095 s
Both perform O(n) insert operations; executemany reduces Python-to-SQLite call overhead.

ADVERSARIAL SENTIMENT TESTS
negative | The air is not good today.
negative | The service was not not good.
negative | 

## Interpretation

The outputs should show that schema and delimiter validation catch incorrect columns, decoding errors are handled, and empty files do not crash the inspection routine. The SQL injection attempt must return no rows when parameterised. The sentiment examples document where the lexicon is unreliable.